# BlindSpotter — MR-GCN Training (Colab)

Binary classification: given a scene-graph frame, predict whether a PM (scooter / cyclist)
hidden in a blind zone will emerge within 3 seconds (label=1) or not (label=0).

**Paper improvements implemented** (Liu et al., *IEEE Transactions on Intelligent Vehicles*, 2023):
1. **Degree Embedding (Eq. 1)** — adds in/out-degree embeddings to node features before message passing (+2% Acc, +1.4% AUC reported in paper)
2. **3-layer MR-GCN** — third `RGCNConv` layer for deeper relation-aware message passing
3. **Combined Readout** — BZ-node embedding concatenated with global mean pool
4. **Tuned Hyper-parameters** — LR=1e-4, weight_decay=5e-4, batch=64

### Running order
1. Run cells **top to bottom** with Shift+Enter
2. In Cell 0: update `DRIVE_DIR` to your own Drive path
3. Training (Cell 8) takes ~5-30 min — use **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
# ── 0. Google Drive mount & path config ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ▼▼▼ Update to your own Drive path ▼▼▼
DRIVE_DIR = '/content/drive/MyDrive/BlindSpotter'
DATA_PKL  = f'{DRIVE_DIR}/graph_dataset.pkl'
SAVE_DIR  = f'{DRIVE_DIR}/outputs'

os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.exists(DATA_PKL):
    print(f'[ERROR] File not found: {DATA_PKL}')
    print('Please update DRIVE_DIR to match your Drive folder.')
else:
    print(f'Data found : {DATA_PKL}')
    print(f'Save dir   : {SAVE_DIR}')
    print(f'File size  : {os.path.getsize(DATA_PKL)/1024/1024:.1f} MB')

In [ ]:
# ── 1. Package installation ────────────────────────────────────────────────────
import subprocess, sys

print('Installing torch_geometric ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'],
    check=True
)
print('Done.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 2. Imports & hyperparameters ───────────────────────────────────────────────
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import RGCNConv, global_mean_pool
from torch_geometric.utils import degree

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, classification_report,
    precision_recall_curve, roc_curve,
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Hyperparameters ────────────────────────────────────────────────────────────
BATCH        = 64       # paper: 64
EPOCHS       = 50
LR           = 1e-4    # paper uses 1e-5; 1e-4 suits our larger dataset
WEIGHT_DECAY = 5e-4   # paper Eq. 5
HIDDEN       = 64
DROPOUT      = 0.3
N_BASES      = 4
MAX_DEG      = 16

print(f'Batch={BATCH}  Epochs={EPOCHS}  LR={LR}  WD={WEIGHT_DECAY}')
print(f'Hidden={HIDDEN}  Dropout={DROPOUT}  N_bases={N_BASES}  Max_deg={MAX_DEG}')

In [ ]:
# ── 3. Relation type mapping ───────────────────────────────────────────────────
RELATION_TYPES = ['spatial_near', 'occludes', 'potential_conflict', 'blind_zone_relation']
REL2ID         = {r: i for i, r in enumerate(RELATION_TYPES)}
NUM_RELATIONS  = len(RELATION_TYPES)
print('Relations:', REL2ID)

In [ ]:
# ── 4. Load dataset & print stats ─────────────────────────────────────────────
with open(DATA_PKL, 'rb') as f:
    raw = pickle.load(f)

NODE_DIM = len(raw['node_feature_names'])
EDGE_DIM = len(raw['edge_feature_names'])
print(f'Node feature dim : {NODE_DIM}  {raw["node_feature_names"]}')
print(f'Edge feature dim : {EDGE_DIM}  {raw["edge_feature_names"]}')
print()

for split in ('train', 'val', 'test'):
    samples = raw['dataset'][split]
    labels  = [s['label'] for s in samples]
    c       = Counter(labels)
    print(f'{split:5s}  total={len(samples):5d}  '
          f'pos={c[1]:4d}  neg={c[0]:4d}  '
          f'pos_rate={c[1]/max(len(samples),1):.3f}')

# Class imbalance weight for BCE loss
train_labels = [s['label'] for s in raw['dataset']['train']]
n_pos = max(sum(train_labels), 1)
n_neg = len(train_labels) - n_pos
POS_WEIGHT = torch.tensor([n_neg / n_pos], dtype=torch.float, device=DEVICE)
print(f'\nPOS_WEIGHT = {POS_WEIGHT.item():.3f}')

In [ ]:
# ── 5. Dataset class & DataLoader ─────────────────────────────────────────────
class SceneGraphDataset(Dataset):
    """Wraps pre-built scene-graph samples as PyG Data objects.
    Includes hidden_sec from meta for Early Warning Time analysis."""

    def __init__(self, samples):
        super().__init__()
        self.samples = samples

    def len(self):
        return len(self.samples)

    def get(self, idx):
        s = self.samples[idx]
        return Data(
            x          = torch.tensor(s['x'],          dtype=torch.float),
            edge_index = torch.tensor(s['edge_index'], dtype=torch.long),
            edge_type  = torch.tensor(
                             [REL2ID.get(et, 0) for et in s['edge_type']],
                             dtype=torch.long),
            y          = torch.tensor([s['label']],    dtype=torch.float),
            bz_idx     = torch.tensor(s['bz_node_idx'], dtype=torch.long),
            # hidden_sec: seconds the PM has been hidden (from meta)
            # Lower TP mean = model detects risk earlier in hiding sequence
            hidden_sec = torch.tensor(
                             [s['meta'].get('hidden_sec', 0.0)],
                             dtype=torch.float),
        )


train_ds = SceneGraphDataset(raw['dataset']['train'])
val_ds   = SceneGraphDataset(raw['dataset']['val'])
test_ds  = SceneGraphDataset(raw['dataset']['test'])

# Colab: use num_workers=2 for faster data loading
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

# Verify batch shapes
batch = next(iter(train_loader))
print('x shape       :', batch.x.shape)
print('edge_index    :', batch.edge_index.shape)
print('edge_type     :', batch.edge_type.shape)
print('y             :', batch.y.shape)
print('bz_idx        :', batch.bz_idx.shape)
print('hidden_sec    :', batch.hidden_sec.shape)
print('batch vector  :', batch.batch.shape)

In [ ]:
# ── 6. MR-GCN Model ────────────────────────────────────────────────────────────
# Implements paper contributions from:
#   Liu et al., "Multirelational Graph Convolutional Network for Blind Spot
#   Prediction", IEEE Transactions on Intelligent Vehicles, 2023.

class BlindSpotterRGCN(nn.Module):
    def __init__(self, node_dim, num_relations, hidden=64,
                 dropout=0.3, n_bases=4, max_deg=16):
        super().__init__()
        self.dropout = dropout

        # ── Paper Eq. 1: Degree Embeddings ────────────────────────────────────
        # Embedding tables for in-degree and out-degree of each node.
        # Summed with raw node features BEFORE any GCN layer,
        # giving structural position awareness beyond feature content.
        # Paper reported: +2% Acc, +1.4% AUC from this single addition.
        self.in_deg_emb  = nn.Embedding(max_deg + 1, node_dim)
        self.out_deg_emb = nn.Embedding(max_deg + 1, node_dim)

        # ── Paper contribution: 3-layer MR-GCN ───────────────────────────────
        # Basis decomposition: W_r = sum_b(a_rb * V_b)
        # Shared bases V_b, relation-specific coefficients a_rb.
        self.conv1 = RGCNConv(node_dim, hidden,
                              num_relations=num_relations, num_bases=n_bases)
        self.conv2 = RGCNConv(hidden, hidden,
                              num_relations=num_relations, num_bases=n_bases)
        self.conv3 = RGCNConv(hidden, hidden,
                              num_relations=num_relations, num_bases=n_bases)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)

        # ── Paper contribution: Combined Readout ─────────────────────────────
        # BZ-node embedding: local occlusion context
        # Global mean pool:  overall scene-graph context
        # Concatenating both gives richer scene representation.
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, edge_index, edge_type, bz_idx, batch_vec):
        # ── Degree embedding (Eq. 1) ─────────────────────────────────────────
        row, col = edge_index
        max_emb  = self.in_deg_emb.num_embeddings - 1
        in_deg   = degree(col, num_nodes=x.size(0),
                          dtype=torch.long).clamp(max=max_emb)
        out_deg  = degree(row, num_nodes=x.size(0),
                          dtype=torch.long).clamp(max=max_emb)
        h = x + self.in_deg_emb(in_deg) + self.out_deg_emb(out_deg)

        # ── 3-layer relational message passing ───────────────────────────────
        h = F.relu(self.bn1(self.conv1(h, edge_index, edge_type)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.bn2(self.conv2(h, edge_index, edge_type)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.relu(self.bn3(self.conv3(h, edge_index, edge_type)))

        # ── Combined readout ─────────────────────────────────────────────────
        num_graphs = int(batch_vec.max().item()) + 1
        offsets    = torch.zeros(num_graphs, dtype=torch.long, device=x.device)
        for i in range(1, num_graphs):
            offsets[i] = (batch_vec < i).sum()

        bz_embed     = h[bz_idx + offsets]            # [B, hidden]
        global_embed = global_mean_pool(h, batch_vec)  # [B, hidden]
        combined     = torch.cat([bz_embed, global_embed], dim=-1)  # [B, hidden*2]

        return self.head(combined).squeeze(-1)          # [B] logits


model = BlindSpotterRGCN(
    node_dim      = NODE_DIM,
    num_relations = NUM_RELATIONS,
    hidden        = HIDDEN,
    dropout       = DROPOUT,
    n_bases       = N_BASES,
    max_deg       = MAX_DEG,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# ── 7. Training helpers ────────────────────────────────────────────────────────

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train=True):
    """Run one epoch; return dict with loss, auroc, auprc, f1,
    probs, labels, and hidden_sec for EWT analysis."""
    model.train() if train else model.eval()
    total_loss = 0.0
    all_logits, all_labels, all_hsec = [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch  = batch.to(DEVICE)
            logits = model(
                batch.x, batch.edge_index, batch.edge_type,
                batch.bz_idx, batch.batch
            )
            loss = criterion(logits, batch.y.view(-1))

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss  += loss.item() * batch.num_graphs
            all_logits.append(logits.detach().cpu())
            all_labels.append(batch.y.view(-1).cpu())
            all_hsec.append(batch.hidden_sec.view(-1).cpu())

    probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)
    hsec   = torch.cat(all_hsec).numpy()
    preds  = (probs >= 0.5).astype(int)
    n      = max(len(labels), 1)

    auroc = roc_auc_score(labels, probs)           if labels.sum() > 0 else 0.0
    auprc = average_precision_score(labels, probs) if labels.sum() > 0 else 0.0
    f1    = f1_score(labels, preds, zero_division=0)

    return dict(
        loss       = total_loss / n,
        auroc      = auroc,
        auprc      = auprc,
        f1         = f1,
        probs      = probs,
        labels     = labels,
        hidden_sec = hsec,
    )


print('Helpers ready.')

In [ ]:
# ── 8. Training loop ───────────────────────────────────────────────────────────
# GPU: ~5-10 min  |  CPU: ~20-40 min

history    = {'train': [], 'val': []}
best_auprc = -1.0
best_ckpt  = f'{SAVE_DIR}/best_mrgcn.pt'

print(f'Starting training ({EPOCHS} epochs) ...\n')

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train'].append(tr)
    history['val'].append(vl)

    # Save best model by validation AUPRC
    if vl['auprc'] > best_auprc:
        best_auprc = vl['auprc']
        torch.save(model.state_dict(), best_ckpt)
        mark = ' *'
    else:
        mark = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train loss={tr["loss"]:.4f} auroc={tr["auroc"]:.4f} auprc={tr["auprc"]:.4f} | '
              f'Val   loss={vl["loss"]:.4f} auroc={vl["auroc"]:.4f} auprc={vl["auprc"]:.4f}{mark}')

print(f'\nBest val AUPRC: {best_auprc:.4f}  ->  saved to {best_ckpt}')

In [ ]:
# ── 9. Plot training curves ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics   = [('loss', 'Loss'), ('auroc', 'AUROC'), ('auprc', 'AUPRC')]

for ax, (key, label) in zip(axes, metrics):
    ax.plot([d[key] for d in history['train']], label='Train', lw=2)
    ax.plot([d[key] for d in history['val']],   label='Val',   lw=2)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('MR-GCN Training Curves (Liu et al. 2023 improvements)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
curve_path = f'{SAVE_DIR}/training_curves.png'
plt.savefig(curve_path, bbox_inches='tight')
plt.show()
print(f'Saved: {curve_path}')

In [ ]:
# ── 10. Full test evaluation ───────────────────────────────────────────────────

# Load best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

# Collect all predictions on test set
all_logits_test, all_labels_test, all_hsec_test = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch  = batch.to(DEVICE)
        logits = model(
            batch.x, batch.edge_index, batch.edge_type,
            batch.bz_idx, batch.batch
        )
        all_logits_test.append(logits.cpu())
        all_labels_test.append(batch.y.view(-1).cpu())
        all_hsec_test.append(batch.hidden_sec.view(-1).cpu())

test_probs  = torch.sigmoid(torch.cat(all_logits_test)).numpy()
test_labels = torch.cat(all_labels_test).numpy().astype(int)
test_hsec   = torch.cat(all_hsec_test).numpy()
test_preds  = (test_probs >= 0.5).astype(int)

# ── Core metrics ──────────────────────────────────────────────────────────────
auroc = roc_auc_score(test_labels, test_probs)
auprc = average_precision_score(test_labels, test_probs)
f1    = f1_score(test_labels, test_preds, zero_division=0)

# ── Recall@FPR ────────────────────────────────────────────────────────────────
fpr_arr, tpr_arr, thr_arr = roc_curve(test_labels, test_probs)

idx_05           = np.searchsorted(fpr_arr, 0.05)
recall_at_fpr05  = tpr_arr[min(idx_05, len(tpr_arr) - 1)]

idx_10           = np.searchsorted(fpr_arr, 0.10)
recall_at_fpr10  = tpr_arr[min(idx_10, len(tpr_arr) - 1)]

# ── Early Warning Time (EWT) ──────────────────────────────────────────────────
# TP: label=1, pred=1  |  FN: label=1, pred=0
# Lower mean TP hidden_sec means model detects risk EARLIER in hiding sequence
TP_mask = (test_labels == 1) & (test_preds == 1)
FN_mask = (test_labels == 1) & (test_preds == 0)
FP_mask = (test_labels == 0) & (test_preds == 1)
TN_mask = (test_labels == 0) & (test_preds == 0)

ewt_tp = test_hsec[TP_mask].mean() if TP_mask.sum() > 0 else float('nan')
ewt_fn = test_hsec[FN_mask].mean() if FN_mask.sum() > 0 else float('nan')

# ── Print results ─────────────────────────────────────────────────────────────
print('=' * 60)
print('TEST SET RESULTS (best val-AUPRC checkpoint)')
print('=' * 60)
print(f'  AUROC              : {auroc:.4f}')
print(f'  AUPRC              : {auprc:.4f}')
print(f'  F1 (@thr=0.5)      : {f1:.4f}')
print('-' * 60)
print(f'  Recall@FPR=0.05    : {recall_at_fpr05:.4f}')
print(f'  Recall@FPR=0.10    : {recall_at_fpr10:.4f}')
print('-' * 60)
print(f'  EWT TP mean hidden_sec : {ewt_tp:.3f} s  (lower = earlier detection)')
print(f'  EWT FN mean hidden_sec : {ewt_fn:.3f} s')
print(f'  TP={TP_mask.sum()}  FP={FP_mask.sum()}  '
      f'TN={TN_mask.sum()}  FN={FN_mask.sum()}')
print('=' * 60)
print()
print(classification_report(test_labels, test_preds,
                            target_names=['No-Emerge', 'Emerge']))

In [ ]:
# ── 11. PR curve & ROC curve ───────────────────────────────────────────────────

prec_arr, rec_arr, _ = precision_recall_curve(test_labels, test_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Precision-Recall curve
ax1.plot(rec_arr, prec_arr, lw=2, label=f'AUPRC={auprc:.3f}')
ax1.fill_between(rec_arr, prec_arr, alpha=0.1)
ax1.axhline(test_labels.mean(), ls='--', color='gray',
            label=f'Baseline (prior={test_labels.mean():.3f})')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ROC curve with FPR operating point markers
ax2.plot(fpr_arr, tpr_arr, lw=2, label=f'AUROC={auroc:.3f}')
ax2.plot([0, 1], [0, 1], ls='--', color='gray', label='Random')
for target_fpr, recall_val, color in [
    (0.05, recall_at_fpr05, 'red'),
    (0.10, recall_at_fpr10, 'orange'),
]:
    ax2.scatter([target_fpr], [recall_val], color=color, zorder=5,
                label=f'FPR={target_fpr:.2f} -> TPR={recall_val:.3f}')
    ax2.axvline(target_fpr, ls=':', color=color, alpha=0.5)
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.set_title('ROC Curve (with FPR operating points)', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig.suptitle('MR-GCN Test Set Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
pr_roc_path = f'{SAVE_DIR}/test_pr_roc.png'
plt.savefig(pr_roc_path, bbox_inches='tight')
plt.show()
print(f'Saved: {pr_roc_path}')

In [ ]:
# ── 12. Early Warning Time (EWT) Analysis ─────────────────────────────────────
# Histogram of hidden_sec by prediction outcome (TP / FP / TN / FN)

outcomes = {
    'TP (Emerge, predicted Emerge)' : test_hsec[TP_mask],
    'FN (Emerge, missed)'           : test_hsec[FN_mask],
    'FP (No-Emerge, false alarm)'   : test_hsec[FP_mask],
    'TN (No-Emerge, correct)'       : test_hsec[TN_mask],
}
colors = ['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histograms by outcome
ax = axes[0]
for (label, vals), col in zip(outcomes.items(), colors):
    if len(vals) > 0:
        ax.hist(vals, bins=20, alpha=0.5,
                label=f'{label} (n={len(vals)})',
                color=col, density=True)
ax.set_xlabel('hidden_sec (seconds)')
ax.set_ylabel('Density')
ax.set_title('Distribution of hidden_sec by Prediction Outcome',
             fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Right: boxplot — positive-class only (TP vs FN)
ax2 = axes[1]
data_box = [test_hsec[TP_mask], test_hsec[FN_mask]]
bp = ax2.boxplot(data_box, labels=['TP\n(detected)', 'FN\n(missed)'],
                 patch_artist=True,
                 medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor('#2ca02c')
bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#d62728')
bp['boxes'][1].set_alpha(0.6)
ax2.set_ylabel('hidden_sec (seconds)')
ax2.set_title('EWT: hidden_sec for TP vs FN', fontweight='bold')
ax2.text(0.5, 0.95,
         f'TP mean={ewt_tp:.2f}s   FN mean={ewt_fn:.2f}s\n'
         'Lower TP mean = earlier detection',
         transform=ax2.transAxes, ha='center', va='top', fontsize=9,
         bbox=dict(boxstyle='round', fc='wheat', alpha=0.6))
ax2.grid(True, alpha=0.3)

fig.suptitle('Early Warning Time (EWT) Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
ewt_path = f'{SAVE_DIR}/ewt_analysis.png'
plt.savefig(ewt_path, bbox_inches='tight')
plt.show()
print(f'Saved: {ewt_path}')

print('\nEWT summary:')
for lbl, vals in outcomes.items():
    if len(vals) > 0:
        print(f'  {lbl:40s}  n={len(vals):4d}  '
              f'mean={vals.mean():.3f}s  median={np.median(vals):.3f}s')

In [ ]:
# ── 13. Threshold Analysis ─────────────────────────────────────────────────────

thresholds = np.linspace(0.01, 0.99, 200)
f1_scores  = [
    f1_score(test_labels, (test_probs >= t).astype(int), zero_division=0)
    for t in thresholds
]
recalls    = [
    ((test_labels == 1) & ((test_probs >= t))).sum() / max(test_labels.sum(), 1)
    for t in thresholds
]

best_f1_thr = thresholds[np.argmax(f1_scores)]
best_f1_val = max(f1_scores)

# Threshold where Recall >= 0.85
recall_arr = np.array(recalls)
valid      = np.where(recall_arr >= 0.85)[0]
rec85_thr  = thresholds[valid[-1]] if len(valid) > 0 else None
rec85_f1   = f1_scores[valid[-1]]  if len(valid) > 0 else None

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thresholds, f1_scores, lw=2)
ax1.axvline(best_f1_thr, ls='--', color='red',
            label=f'Best F1={best_f1_val:.3f} @ thr={best_f1_thr:.2f}')
ax1.axvline(0.5, ls=':', color='gray', label='Default thr=0.5')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('F1')
ax1.set_title('F1 vs Threshold', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(thresholds, recall_arr, lw=2)
ax2.axhline(0.85, ls='--', color='orange', label='Recall=0.85')
if rec85_thr is not None:
    ax2.axvline(rec85_thr, ls='--', color='red',
                label=f'thr={rec85_thr:.2f} -> F1={rec85_f1:.3f}')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Recall')
ax2.set_title('Recall vs Threshold', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
thr_path = f'{SAVE_DIR}/threshold_analysis.png'
plt.savefig(thr_path, bbox_inches='tight')
plt.show()
print(f'Saved: {thr_path}')

print(f'\nMax-F1 threshold : {best_f1_thr:.3f}  (F1={best_f1_val:.4f})')
if rec85_thr is not None:
    print(f'Recall>=0.85 thr : {rec85_thr:.3f}  (F1={rec85_f1:.4f})')
else:
    print('Recall>=0.85 not achievable in [0.01, 0.99] range')

In [ ]:
# ── 14. Save model + config + metrics to Drive ─────────────────────────────────
import json

save_payload = {
    'model_state' : model.state_dict(),
    'config': {
        'node_dim'      : NODE_DIM,
        'num_relations' : NUM_RELATIONS,
        'hidden'        : HIDDEN,
        'dropout'       : DROPOUT,
        'n_bases'       : N_BASES,
        'max_deg'       : MAX_DEG,
        'lr'            : LR,
        'weight_decay'  : WEIGHT_DECAY,
        'batch'         : BATCH,
        'epochs'        : EPOCHS,
    },
    'metrics': {
        'test_auroc'         : float(auroc),
        'test_auprc'         : float(auprc),
        'test_f1'            : float(f1),
        'recall_at_fpr05'    : float(recall_at_fpr05),
        'recall_at_fpr10'    : float(recall_at_fpr10),
        'ewt_tp_mean'        : float(ewt_tp),
        'ewt_fn_mean'        : float(ewt_fn),
        'best_f1_threshold'  : float(best_f1_thr),
        'best_f1_value'      : float(best_f1_val),
        'recall85_threshold' : float(rec85_thr) if rec85_thr is not None else None,
    },
    'relation_types'     : RELATION_TYPES,
    'node_feature_names' : raw['node_feature_names'],
    'edge_feature_names' : raw['edge_feature_names'],
}

model_path = f'{SAVE_DIR}/mrgcn_final.pt'
torch.save(save_payload, model_path)
print(f'Saved model : {model_path}')

# Human-readable metrics JSON
metrics_path = f'{SAVE_DIR}/mrgcn_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as fp:
    json.dump(save_payload['metrics'], fp, indent=2, ensure_ascii=False)
print(f'Saved metrics: {metrics_path}')
print()
print(json.dumps(save_payload['metrics'], indent=2, ensure_ascii=False))

## Next Steps

### Temporal Transformer for Sequence Modeling (paper suggestion)

The current model treats each scene-graph frame independently. Liu et al. suggest extending
to a **Temporal Graph Transformer** that processes a sliding window of consecutive frames:

```
frame_t-2 -> MR-GCN -> h_t-2 -+
frame_t-1 -> MR-GCN -> h_t-1 --+-> Transformer Encoder -> classifier
frame_t   -> MR-GCN -> h_t   -+
```

**Implementation sketch:**
```python
class TemporalBlindSpotter(nn.Module):
    def __init__(self, ...):
        self.rgcn    = BlindSpotterRGCN(...)    # shared weights
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=HIDDEN*2, nhead=4),
            num_layers=2
        )
        self.head = nn.Linear(HIDDEN*2, 1)

    def forward(self, frame_sequence):
        # frame_sequence: list of T PyG batches
        h_seq = [self.rgcn(...frame...) for frame in frame_sequence]  # [T, B, H]
        h_seq = torch.stack(h_seq)              # [T, B, H]
        out   = self.encoder(h_seq)             # [T, B, H]
        return self.head(out[-1]).squeeze(-1)   # use last frame representation
```

Expected gain (from paper): +1-2% AUROC on temporal sequences of >= 3 frames.